# Coupled Cluster Doubles (CCD) for the Pairing ModelThis notebook implements and compares three methods:1. **Exact Diagonalization** - Full numerical solution2. **MBPT2** - Many-Body Perturbation Theory (2nd order)3. **CCD** - Coupled Cluster Doubles## System Parameters- **4 fermions** in **4 doubly degenerate levels**- Reference state: pairs in lowest levels (0,1)- Basis dimension: C(4,2) = **6 states**## Important NoteSingle-particle energies include **mean-field pairing** from the Fock matrix.

In [ ]:
# Importsfrom functools import partialimport jaximport jax.numpy as jnpimport numpy as npfrom jax import Array, jitimport matplotlib.pyplot as pltfrom itertools import combinationsfrom scipy.linalg import eighjax.config.update("jax_enable_x64", True)print("✓ Imports complete")

## 1. Exact Diagonalization (CORRECTED)**Key correction**: Single-particle energies must match the Fock matrix.

In [ ]:
def exact_diagonalization_pairing(pnum, hnum, delta, g):    """Exact diagonalization with correct single-particle energies."""    n_levels = 4    n_pairs = 2        basis_states = list(combinations(range(n_levels), n_pairs))    n_basis = len(basis_states)        # CORRECTED: Single-particle energies from Fock matrix    deltaval = 0.5 * delta    gval = -0.5 * g        epsilon = np.zeros(n_levels)    epsilon[0] = deltaval * 0 + gval    epsilon[1] = deltaval * 2 + gval    epsilon[2] = deltaval * (hnum + 0)    epsilon[3] = deltaval * (hnum + 2)        H = np.zeros((n_basis, n_basis))        for i, state_i in enumerate(basis_states):        for j, state_j in enumerate(basis_states):            if i == j:                H[i, j] = sum(2 * epsilon[k] for k in state_i)                        for k in range(n_levels):                for l in range(n_levels):                    if l in state_j and k not in state_j:                        state_temp = list(state_j)                        state_temp.remove(l)                        state_temp.append(k)                        state_temp = tuple(sorted(state_temp))                        if state_temp == state_i:                            H[i, j] -= g / 2.0        eigenvalues, eigenvectors = eigh(H)    return eigenvalues, eigenvectors, basis_states, Hprint("✓ Exact diagonalization defined")

## 2. CCD Implementation (Original Code)

In [ ]:
def init_pairing_v(g, pnum, hnum):    v_pppp = np.zeros((pnum, pnum, pnum, pnum))    v_pphh = np.zeros((pnum, pnum, hnum, hnum))    v_hhhh = np.zeros((hnum, hnum, hnum, hnum))    gval = -0.5 * g    p_even = np.arange(0, pnum, 2)    h_even = np.arange(0, hnum, 2)    for a in p_even:        for b in p_even:            v_pppp[a, a + 1, b, b + 1] = gval            v_pppp[a + 1, a, b, b + 1] = -gval            v_pppp[a, a + 1, b + 1, b] = -gval            v_pppp[a + 1, a, b + 1, b] = gval    for a in p_even:        for i in h_even:            v_pphh[a, a + 1, i, i + 1] = gval            v_pphh[a + 1, a, i, i + 1] = -gval            v_pphh[a, a + 1, i + 1, i] = -gval            v_pphh[a + 1, a, i + 1, i] = gval    for j in h_even:        for i in h_even:            v_hhhh[j, j + 1, i, i + 1] = gval            v_hhhh[j + 1, j, i, i + 1] = -gval            v_hhhh[j, j + 1, i + 1, i] = -gval            v_hhhh[j + 1, j, i + 1, i] = gval    v_pppp, v_pphh, v_hhhh = map(jnp.array, (v_pppp, v_pphh, v_hhhh))    return v_pppp, v_pphh, v_hhhhdef init_pairing_fock(delta, g, pnum, hnum):    deltaval = 0.5 * delta    gval = -0.5 * g    f_pp = np.zeros((pnum, pnum))    f_hh = np.zeros((hnum, hnum))    i_even = np.arange(0, hnum, 2)    f_hh[i_even, i_even] = deltaval * i_even + gval    f_hh[i_even + 1, i_even + 1] = deltaval * i_even + gval    a_even = np.arange(0, pnum, 2)    f_pp[a_even, a_even] = deltaval * (hnum + a_even)    f_pp[a_even + 1, a_even + 1] = deltaval * (hnum + a_even)    f_pp, f_hh = jnp.array(f_pp), jnp.array(f_hh)    return f_pp, f_hhdef init_t2(v_pphh, f_pp, f_hh):    Fhh_diag = jnp.diag(f_hh)    Fpp_diag = jnp.diag(f_pp)    denom = (Fhh_diag[None, None, :, None] + Fhh_diag[None, None, None, :]              - Fpp_diag[:, None, None, None] - Fpp_diag[None, :, None, None])    return v_pphh / denom@jitdef ccd_energy(v_pphh, t2):    return 0.25 * jnp.sum(v_pphh * t2)print("✓ CCD helper functions defined")

In [ ]:
@jitdef ccd_iter(v_pppp, v_pphh, v_hhhh, f_pp, f_hh, t2):    Hbar_pphh = (v_pphh + jnp.einsum("bc,acij->abij", f_pp, t2)                 - jnp.einsum("ac,bcij->abij", f_pp, t2)                 - jnp.einsum("abik,kj->abij", t2, f_hh)                 + jnp.einsum("abjk,ki->abij", t2, f_hh)                 + 0.5 * jnp.einsum("abcd,cdij->abij", v_pppp, t2)                 + 0.5 * jnp.einsum("abkl,klij->abij", t2, v_hhhh))    chi_hh = 0.5 * jnp.einsum("cdkl,cdjl->kj", v_pphh, t2)    Hbar_pphh -= jnp.einsum("abik,kj->abij", t2, chi_hh) - jnp.einsum("abik,kj->abji", t2, chi_hh)    chi_pp = -0.5 * jnp.einsum("cdkl,bdkl->cb", v_pphh, t2)    Hbar_pphh += jnp.einsum("acij,cb->abij", t2, chi_pp) - jnp.einsum("acij,cb->baij", t2, chi_pp)    chi_hhhh = 0.5 * jnp.einsum("cdkl,cdij->klij", v_pphh, t2)    Hbar_pphh += 0.5 * jnp.einsum("abkl,klij->abij", t2, chi_hhhh)    chi_phph = 0.5 * jnp.einsum("cdkl,dblj->bkcj", v_pphh, t2)    Hbar_pphh += (jnp.einsum("bkcj,acik->abij", chi_phph, t2)                  - jnp.einsum("bkcj,acik->baij", chi_phph, t2)                  - jnp.einsum("bkcj,acik->abji", chi_phph, t2)                  + jnp.einsum("bkcj,acik->baji", chi_phph, t2))    Fhh_diag = jnp.diag(f_hh)    Fpp_diag = jnp.diag(f_pp)    denom = (Fhh_diag[None, None, :, None] + Fhh_diag[None, None, None, :]             - Fpp_diag[:, None, None, None] - Fpp_diag[None, :, None, None])    return t2 + Hbar_pphh / denomdef calculate_ccd_energy(pnum, hnum, delta, g, verbose=False):    v_pppp, v_pphh, v_hhhh = init_pairing_v(g, pnum, hnum)    f_pp, f_hh = init_pairing_fock(delta, g, pnum, hnum)    t2 = init_t2(v_pphh, f_pp, f_hh)    mbpt2_erg = ccd_energy(v_pphh, t2)    niter, mix, erg_old, eps = 200, 0.3, 0.0, jnp.float64(1.0e-10)    for iter_count in range(niter):        t2_new = ccd_iter(v_pppp, v_pphh, v_hhhh, f_pp, f_hh, t2)        erg = ccd_energy(v_pphh, t2_new)        if jnp.abs(erg - erg_old) / jnp.abs(erg) < eps:            if verbose:                print(f"CCD converged in {iter_count} iterations")            break        erg_old = erg        t2 = mix * t2_new + (1.0 - mix) * t2    if verbose:        print(f"MBPT2 correlation: {mbpt2_erg:.10f}")        print(f"CCD correlation:   {erg:.10f}")    return float(erg), float(mbpt2_erg)print("✓ CCD solver defined")

## 3. Single Point Comparison (g=0.5)

In [ ]:
pnum, hnum, delta, g_test = 4, 4, 1.0, 0.5print("="*80)print("SINGLE POINT COMPARISON AT g = 0.5")print("="*80)eigenvalues, eigenvectors, basis_states, H = exact_diagonalization_pairing(pnum, hnum, delta, g_test)print(f"\nBasis dimension: {len(basis_states)}")print(f"Basis states: {basis_states}")print(f"\nHamiltonian ({H.shape[0]}×{H.shape[1]}):")print(H)print(f"\nEigenvalues:")for i, E in enumerate(eigenvalues):    print(f"  E_{i}: {E:+.10f}")E_exact = eigenvalues[0]E_ref = H[0, 0]E_ccd_corr, E_mbpt2_corr = calculate_ccd_energy(pnum, hnum, delta, g_test, verbose=True)print(f"\n{'='*80}")print("COMPARISON")print("="*80)print(f"\n{'Method':15s} | {'Total E':>14s} | {'Corr E':>14s} | {'Error':>12s}")print("-"*70)print(f"{'Exact':15s} | {E_exact:+14.10f} | {E_exact-E_ref:+14.10f} | {'--':>12s}")print(f"{'MBPT2':15s} | {E_ref+E_mbpt2_corr:+14.10f} | {E_mbpt2_corr:+14.10f} | {abs(E_ref+E_mbpt2_corr-E_exact):12.2e}")print(f"{'CCD':15s} | {E_ref+E_ccd_corr:+14.10f} | {E_ccd_corr:+14.10f} | {abs(E_ref+E_ccd_corr-E_exact):12.2e}")

## 4. Energy Scan vs g

In [ ]:
g_values = np.linspace(-1, 1, 50)energies_exact = np.zeros_like(g_values)energies_ccd = np.zeros_like(g_values)energies_mbpt2 = np.zeros_like(g_values)print("Computing energies...")for i, g in enumerate(g_values):    evals, _, _, H_mat = exact_diagonalization_pairing(pnum, hnum, delta, g)    energies_exact[i] = evals[0]    E_ccd_c, E_mbpt2_c = calculate_ccd_energy(pnum, hnum, delta, g)    E_ref = H_mat[0, 0]    energies_ccd[i] = E_ref + E_ccd_c    energies_mbpt2[i] = E_ref + E_mbpt2_c    if (i+1) % 10 == 0:        print(f"  {i+1}/{len(g_values)}")print("✓ Complete")

## 5. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))ax1 = axes[0, 0]ax1.plot(g_values, energies_exact, 'k-', lw=2.5, label='Exact')ax1.plot(g_values, energies_ccd, 'b--', lw=2, label='CCD')ax1.plot(g_values, energies_mbpt2, 'r:', lw=2, label='MBPT2')ax1.set_xlabel('g'); ax1.set_ylabel('Energy')ax1.set_title('Energy vs g', fontweight='bold')ax1.legend(); ax1.grid(alpha=0.3)ax2 = axes[0, 1]error_ccd = np.abs(energies_ccd - energies_exact)error_mbpt2 = np.abs(energies_mbpt2 - energies_exact)ax2.semilogy(g_values, error_mbpt2, 'r-', lw=2, label='MBPT2')ax2.semilogy(g_values, error_ccd, 'b-', lw=2, label='CCD')ax2.set_xlabel('g'); ax2.set_ylabel('|E - E_exact|')ax2.set_title('Absolute Error', fontweight='bold')ax2.legend(); ax2.grid(alpha=0.3, which='both')ax3 = axes[1, 0]ax3.plot(g_values, energies_exact - energies_mbpt2, 'r-', lw=2, label='E_exact - E_MBPT2')ax3.plot(g_values, energies_exact - energies_ccd, 'b-', lw=2, label='E_exact - E_CCD')ax3.axhline(0, color='k', ls='--', alpha=0.5)ax3.set_xlabel('g'); ax3.set_ylabel('Difference')ax3.set_title('Energy Differences', fontweight='bold')ax3.legend(); ax3.grid(alpha=0.3)ax4 = axes[1, 1]improvement = error_mbpt2 / (error_ccd + 1e-12)ax4.plot(g_values, improvement, 'g-', lw=2)ax4.axhline(1, color='k', ls='--', alpha=0.5)ax4.set_xlabel('g'); ax4.set_ylabel('Error(MBPT2) / Error(CCD)')ax4.set_title('CCD Improvement Factor', fontweight='bold')ax4.grid(alpha=0.3)plt.suptitle('Pairing Model: Exact vs MBPT2 vs CCD', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('pairing_comparison.png', dpi=150, bbox_inches='tight')plt.show()print("✓ Plots saved")

## 6. Summary Statistics

In [ ]:
print("="*80)print("STATISTICS")print("="*80)print(f"\n{'Metric':25s} | {'MBPT2':>15s} | {'CCD':>15s}")print("-"*65)print(f"{'Max error':25s} | {np.max(error_mbpt2):15.6e} | {np.max(error_ccd):15.6e}")print(f"{'Mean error':25s} | {np.mean(error_mbpt2):15.6e} | {np.mean(error_ccd):15.6e}")print(f"{'RMS error':25s} | {np.sqrt(np.mean(error_mbpt2**2)):15.6e} | {np.sqrt(np.mean(error_ccd**2)):15.6e}")improvement = np.mean(error_mbpt2) / np.mean(error_ccd)print(f"\nCCD is {improvement:.1f}× more accurate than MBPT2")